# Stroke Prevention Demo Final Baseline Model (Logistic Regression)

DP1 Contributor: Erik Bergmark (erikcb2)  
DP2 Contributor: Joshua Lee (jcl12)  
Consolidated by: Daryl Okeke (dokek2)

## Purpose
This notebook trains a baseline model to predict `stroke` using a clean workflow:
- Drops known leakage columns
- Uses a fixed train test split so everyone gets the same results
- Trains logistic regression with class imbalance handling
- Evaluates on the test set only
- Produces a risk score (probability) plus a simple demo threshold for yes or no

## How to run
Run cells from top to bottom.

## What most people will tweak
Go to Section 1 Config. Most changes should be made there.


## 1. Config

This section is the main place to tweak the baseline without touching the rest.

Key ideas:
- `TEST_SIZE` and `RANDOM_STATE` control the split. Do not change these unless a ticket tells you to.
- `LEAKAGE_COLS` are columns we remove because they can act like post stroke proxy signals.
- `CODED_CATEGORICAL_COLS` are columns that look numeric but are really categories. We one hot encode them.
- `THRESHOLD` controls how strict we are when turning a probability into a yes or no prediction for metrics.


In [ ]:
from pathlib import Path

# Split settings (keep fixed for reproducibility)
TEST_SIZE = 0.20
RANDOM_STATE = 42
TRY_STRATIFY = True

# Demo threshold for turning probability into 0 or 1 predictions
# Lower threshold catches more stroke cases but creates more false alarms
THRESHOLD = 0.30

# Data location (relative to repo root)
DATA_REL_PATH = Path("data/raw/stroke_data.csv")

# Output report location (relative to repo root)
REPORT_REL_PATH = Path("reports/baseline_metrics.md")

# Known leakage columns to drop (only dropped if they exist in the dataset)
LEAKAGE_COLS = [
    "General health condition",
    "depression",
    "Minutes sedentary activity",
]

# Columns that are coded categories even though they are numbers
# Column names are stripped of whitespace in Section 3b before these are used
CODED_CATEGORICAL_COLS = [
    "gender",
    "age",
    "Race",
    "Marital status",
    "alcohol",
    "smoke",
    "sleep disorder",
    "Health Insurance",
    "diabetes",
    "hypertension",
    "high cholesterol",
    "Coronary Heart Disease",
    "Body Mass Index",
]

# Dietary/nutritional columns where a recorded value of 0 is physiologically implausible
# A full-day dietary recall of zero calories, protein, fat, etc. indicates a missing response
# These zeros are replaced with NaN before the split so the imputer handles them properly
ZERO_AS_MISSING_COLS = [
    "energy",
    "protein",
    "Carbohydrate",
    "Dietary fiber",
    "Total fat",
    "Total saturated fatty acids",
    "Total monounsaturated fatty acids",
    "Total polyunsaturated fatty acids",
    "Potassium",
    "Sodium",
]

# Outlier clipping settings
# Bounds are computed on the training set only to prevent leakage
# k=3.0 is lenient: only clips extreme outliers, not borderline values
CLIP_OUTLIERS = True
CLIP_IQR_MULTIPLIER = 3.0

# Logistic regression settings
CLASS_WEIGHT = "balanced"
MAX_ITER = 2000
SOLVER = "liblinear"

# Thresholds we quickly scan for a precision/recall tradeoff check
THRESHOLD_SCAN = [0.05, 0.10, 0.20, 0.30]

## 2. Imports

In [2]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


## 3. Load data

This notebook expects the CSV at `data/raw/stroke_data.csv` in the repo.

We find the repo root by searching upward for that file.


In [3]:
def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / DATA_REL_PATH).exists():
            return p
    return here

ROOT = find_repo_root()
DATA_PATH = ROOT / DATA_REL_PATH

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find dataset at: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
df.head()


,stroke,gender,age,Race,Marital status,alcohol,smoke,sleep disorder,Health Insurance,General health condition,...,energy,protein,Carbohydrate,Dietary fiber,Total fat,Total saturated fatty acids,Total monounsaturated fatty acids,Total polyunsaturated fatty acids,Potassium,Sodium
0,0,2,2,5,1,0,0,2,2,3,...,1598,62.78,192.19,10.0,65.64,25.112,24.090,8.543,2887,2969
1,0,2,2,1,1,0,0,1,2,3,...,1547,45.35,256.02,17.0,42.56,13.423,15.389,10.613,2058,2091
2,1,1,2,3,1,1,1,2,1,3,...,2466,81.56,254.49,13.0,103.32,43.295,36.727,15.366,3117,5233
3,0,2,3,3,1,1,1,2,1,4,...,1605,70.99,143.37,10.0,81.60,24.527,30.567,18.174,1766,3706
4,0,1,1,4,1,0,0,2,1,2,...,1818,74.75,229.45,14.2,67.49,26.030,24.837,10.533,1842,2461


## 3b. Data cleaning

Steps applied before splitting:
- Strip whitespace from all column names (the raw CSV has `alcohol ` with a trailing space)
- Check for and drop duplicate rows
- Replace zeros in dietary columns with NaN — zero daily intake of calories, fat, protein, etc. is physiologically implausible and indicates a missing survey response, not a true zero
- Print a data quality summary so any remaining edge cases are visible

In [ ]:
# --- Strip whitespace from column names ---
original_cols = df.columns.tolist()
df.columns = df.columns.str.strip()
changed = [(o, n) for o, n in zip(original_cols, df.columns.tolist()) if o != n]
if changed:
    print("Column names stripped (trailing/leading whitespace removed):")
    for old, new in changed:
        print(f"  '{old}'  ->  '{new}'")
else:
    print("No column name whitespace issues found.")

# --- Duplicate row check ---
n_dupes = df.duplicated().sum()
print(f"\nDuplicate rows found: {n_dupes}")
if n_dupes > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Dropped {n_dupes} duplicate rows. Dataset now has {len(df)} rows.")

In [ ]:
# --- Replace 0 with NaN in dietary columns ---
# A full-day dietary recall reporting zero calories, fat, protein, etc. means
# the participant did not complete that section, not that they consumed nothing.
# Replacing with NaN lets the median imputer fill these in during preprocessing.
zero_cols_present = [c for c in ZERO_AS_MISSING_COLS if c in df.columns]
zero_counts = (df[zero_cols_present] == 0).sum()

df[zero_cols_present] = df[zero_cols_present].replace(0, np.nan)

print("Zeros replaced with NaN in dietary columns:")
print(f"{'Column':<45} {'Values replaced':>15}")
print("-" * 62)
any_replaced = False
for col in zero_cols_present:
    n = zero_counts[col]
    if n > 0:
        print(f"{col:<45} {n:>15}")
        any_replaced = True
if not any_replaced:
    print("  None found.")
print(f"\nTotal values replaced: {zero_counts.sum()}")
print("These will be filled by median imputation in the preprocessing pipeline.")

# --- Data quality summary ---
print("\nData quality summary (numeric columns, excluding target):")
numeric_df = df.select_dtypes(include="number").drop(columns=["stroke"], errors="ignore")
summary = pd.DataFrame({
    "missing": numeric_df.isna().sum(),
    "min": numeric_df.min(),
    "max": numeric_df.max(),
    "median": numeric_df.median(),
}).sort_values("missing", ascending=False)
print(summary.to_string())

## 4. Define label and features

- `y` is the label we want to predict: `stroke`
- `X` is everything else

We also drop leakage columns here before splitting so they never reach the model.

We print label balance because stroke is rare. That is why accuracy can be misleading.


In [4]:
if "stroke" not in df.columns:
    raise ValueError("Dataset must contain a 'stroke' column.")

# Drop leakage columns if present
drop_cols = [c for c in LEAKAGE_COLS if c in df.columns]
df_clean = df.drop(columns=drop_cols)

y = df_clean["stroke"]
X = df_clean.drop(columns=["stroke"])

print("Dropped leakage columns:", drop_cols)
print()
print("Label counts")
print(y.value_counts(dropna=False))
print()
print("Label proportions")
print(y.value_counts(normalize=True, dropna=False))


Dropped leakage columns: ['General health condition', 'depression', 'Minutes sedentary activity']

Label counts
stroke
0    4241
1     362
Name: count, dtype: int64

Label proportions
stroke
0    0.921356
1    0.078644
Name: proportion, dtype: float64


## 5. Train test split

We split once and keep it fixed so everyone gets the same results.

- Train set is what the model learns from
- Test set is what we evaluate on

We use stratify when possible so stroke percent is similar in train and test.


In [5]:
try:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y if TRY_STRATIFY else None,
    )
    stratify_used = "yes" if TRY_STRATIFY else "no"
    stratify_note = ""
except ValueError as e:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=None,
    )
    stratify_used = "no"
    stratify_note = str(e)

print(f"Split standardized: test_size={TEST_SIZE}, seed={RANDOM_STATE}, stratify={stratify_used}")
if stratify_note:
    print("Stratify note")
    print(stratify_note)


Split standardized: test_size=0.2, seed=42, stratify=yes


## 5b. Outlier clipping

Extreme values in numeric columns (e.g. Sodium=7, very high triglycerides) can distort StandardScaler and destabilize logistic regression.

We use IQR-based clipping:
- Bounds are computed on the **training set only** to prevent leakage into the test set
- `lower = Q1 - k * IQR`, `upper = Q3 + k * IQR`
- `k = CLIP_IQR_MULTIPLIER` (default 3.0 — lenient, only clips the most extreme values)

Categorical columns are not clipped.

In [ ]:
if CLIP_OUTLIERS:
    # Identify numeric columns (everything that is not a coded categorical)
    clip_cat_cols = [c for c in CODED_CATEGORICAL_COLS if c in X_train.columns]
    clip_num_cols = [c for c in X_train.columns if c not in clip_cat_cols]

    # Compute IQR bounds from training set only
    Q1 = X_train[clip_num_cols].quantile(0.25)
    Q3 = X_train[clip_num_cols].quantile(0.75)
    IQR = Q3 - Q1
    lower_bounds = Q1 - CLIP_IQR_MULTIPLIER * IQR
    upper_bounds = Q3 + CLIP_IQR_MULTIPLIER * IQR

    # Count how many values fall outside bounds in train (for transparency)
    clipped_low = (X_train[clip_num_cols] < lower_bounds).sum()
    clipped_high = (X_train[clip_num_cols] > upper_bounds).sum()
    total_clipped = (clipped_low + clipped_high).sum()

    # Apply clipping to both train and test using train-derived bounds
    X_train = X_train.copy()
    X_test = X_test.copy()
    X_train[clip_num_cols] = X_train[clip_num_cols].clip(lower=lower_bounds, upper=upper_bounds, axis=1)
    X_test[clip_num_cols] = X_test[clip_num_cols].clip(lower=lower_bounds, upper=upper_bounds, axis=1)

    print(f"Outlier clipping applied (k={CLIP_IQR_MULTIPLIER})")
    print(f"Total values clipped in training set: {total_clipped}")

    affected = [(col, clipped_low[col], clipped_high[col])
                for col in clip_num_cols
                if clipped_low[col] > 0 or clipped_high[col] > 0]
    if affected:
        print(f"\n{'Column':<45} {'Clipped low':>11} {'Clipped high':>12}")
        print("-" * 70)
        for col, lo, hi in affected:
            print(f"{col:<45} {lo:>11} {hi:>12}")
    else:
        print("No values were outside the clip bounds.")
else:
    print("Outlier clipping skipped (CLIP_OUTLIERS=False in config).")

## 6. Preprocessing

Why this section exists:
Some columns are coded categories like `Race` or `Marital status`. They are numbers but they are not real numeric scales.
We treat those as categories and convert them into one hot columns.

Steps:
- Categorical columns: impute most common value, then one hot encode
- Numeric columns: impute median, then scale

Scaling helps logistic regression behave nicely and avoids numeric stability issues.


In [6]:
# Only keep coded categorical columns that are actually in the dataset
cat_cols = [c for c in CODED_CATEGORICAL_COLS if c in X.columns]
num_cols = [c for c in X.columns if c not in cat_cols]

print("Categorical columns (coded):", cat_cols)
print("Number of categorical columns:", len(cat_cols))
print("Number of numeric columns:", len(num_cols))

cat_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

num_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("cat", cat_pipe, cat_cols),
        ("num", num_pipe, num_cols),
    ]
)


Categorical columns (coded): ['gender', 'age', 'Race', 'Marital status', 'alcohol ', 'smoke', 'sleep disorder', 'Health Insurance', 'diabetes', 'hypertension', 'high cholesterol', 'Coronary Heart Disease', 'Body Mass Index']
Number of categorical columns: 13
Number of numeric columns: 19


## 7. Model

We use logistic regression as a simple baseline.
We set `class_weight="balanced"` because stroke is rare.
This helps the model pay more attention to the minority class.

This model outputs probabilities with `predict_proba`.
Those probabilities are our risk scores.


In [7]:
model = LogisticRegression(
    max_iter=MAX_ITER,
    solver=SOLVER,
    class_weight=CLASS_WEIGHT,
    random_state=RANDOM_STATE,
)

pipeline = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", model),
    ]
)

pipeline


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers cont

## 8. Train

This fits the preprocessing and the model on the training set only.


In [8]:
pipeline.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers cont

## 9. Evaluate on test set

Important definitions:
- `y_prob` is the predicted probability of stroke. This is the risk score.
- `THRESHOLD` turns probability into a yes or no prediction for metrics.

Metrics in simple language:
- Accuracy: percent correct overall
- Precision: when we flag stroke, how often we are right
- Recall: out of true stroke cases, how many we catch
- ROC AUC: how well the risk scores rank stroke above non stroke across all possible thresholds

Confusion matrix format:
[[TN FP]
 [FN TP]]

For a prevention demo, recall matters a lot. Missing stroke cases is worse than extra false alarms in an educational demo.


In [9]:
# Risk scores on the test set
y_prob = pipeline.predict_proba(X_test)[:, 1]

# Turn risk scores into predictions using the chosen threshold
y_pred = (y_prob >= THRESHOLD).astype(int)

accuracy = float(accuracy_score(y_test, y_pred))
precision = float(precision_score(y_test, y_pred, zero_division=0))
recall = float(recall_score(y_test, y_pred, zero_division=0))
auc = float(roc_auc_score(y_test, y_prob))
cm = confusion_matrix(y_test, y_pred)

print("Threshold", THRESHOLD)
print("Accuracy", accuracy)
print("Precision", precision)
print("Recall", recall)
print("ROC AUC", auc)
print("Confusion matrix")
print(cm)


Threshold 0.3
Accuracy 0.40716612377850164
Precision 0.10367892976588629
Recall 0.8611111111111112
ROC AUC 0.6145465253239104
Confusion matrix
[[313 536]
 [ 10  62]]


## 10. Score distribution cutoffs (for risk labels)

Docs and Content uses the model score distribution to define low, medium, and high risk bins.
We report:
- min and max
- median (50th percentile)
- 80th percentile
- 95th percentile

Percentile meaning:
- 80th percentile means the score is higher than about 80 percent of people in the test set.

These cutoffs are for demo labeling only. They are not clinical thresholds.


In [10]:
cutoffs = {
    "min": float(np.min(y_prob)),
    "median_p50": float(np.percentile(y_prob, 50)),
    "p80": float(np.percentile(y_prob, 80)),
    "p95": float(np.percentile(y_prob, 95)),
    "max": float(np.max(y_prob)),
}

for k, v in cutoffs.items():
    print(f"{k}: {v:.6f}")


min: 0.007715
median_p50: 0.388876
p80: 0.618423
p95: 0.772234
max: 0.925453


## 11. Quick threshold scan (precision and recall tradeoff)

This prints a few thresholds so we can choose a good demo default.
Lower thresholds increase recall and decrease precision.

This does not change ROC AUC because AUC uses the full probability ranking.


In [11]:
for t in THRESHOLD_SCAN:
    yp = (y_prob >= t).astype(int)
    p = float(precision_score(y_test, yp, zero_division=0))
    r = float(recall_score(y_test, yp, zero_division=0))
    c = confusion_matrix(y_test, yp)
    print(f"threshold={t:.2f} | precision={p:.4f} | recall={r:.4f} | cm={c.tolist()}")


threshold=0.05 | precision=0.0793 | recall=1.0000 | cm=[[13, 836], [0, 72]]
threshold=0.10 | precision=0.0824 | recall=0.9861 | cm=[[58, 791], [1, 71]]
threshold=0.20 | precision=0.0901 | recall=0.9306 | cm=[[172, 677], [5, 67]]
threshold=0.30 | precision=0.1037 | recall=0.8611 | cm=[[313, 536], [10, 62]]


## 12. Save report

This writes a single markdown report to keep results easy to find.
It includes the split settings, leakage columns dropped, threshold used, metrics, confusion matrix, and score cutoffs.

File:
- `reports/baseline_metrics.md`


In [12]:
REPORT_PATH = ROOT / REPORT_REL_PATH
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write("# Baseline Metrics (Final Demo Baseline)\n\n")

    f.write("## Model\n")
    f.write("Logistic Regression\n\n")

    f.write("## Split\n")
    f.write(f"test_size: {TEST_SIZE}\n")
    f.write(f"random_state: {RANDOM_STATE}\n")
    f.write(f"stratify: {stratify_used}\n")
    if stratify_note:
        f.write("stratify_note:\n")
        f.write(f"{stratify_note}\n")
    f.write("\n")

    f.write("## Leakage handling\n")
    f.write("Dropped columns\n")
    for c in drop_cols:
        f.write(f"- {c}\n")
    f.write("\n")

    f.write("## Threshold\n")
    f.write(f"threshold: {THRESHOLD}\n\n")

    f.write("## Label balance\n")
    f.write("Counts\n")
    f.write(y.value_counts(dropna=False).to_string())
    f.write("\n\n")

    f.write("## Metrics on test set\n")
    f.write(f"Accuracy: {accuracy:.4f}\n")
    f.write(f"Precision: {precision:.4f}\n")
    f.write(f"Recall: {recall:.4f}\n")
    f.write(f"ROC AUC: {auc:.4f}\n\n")

    f.write("## Confusion matrix on test set\n")
    f.write("Format is [[TN FP]\n")
    f.write("           [FN TP]]\n\n")
    f.write(np.array2string(cm))
    f.write("\n\n")

    f.write("## Score cutoffs from test set probabilities\n")
    for k, v in cutoffs.items():
        f.write(f"- {k}: {v:.6f}\n")

print(f"Saved report to {REPORT_PATH}")


Saved report to C:\Users\dokek\OneDrive\GitHub\HAS\stroke-prevention-demo\reports\baseline_metrics.md


Export the trained pipeline for the app

If you want the Streamlit app to load a trained model without retraining, save the pipeline with joblib.

This creates:
- `models/baseline_pipeline.joblib`

Before running, make sure repo plan allows committing or storing model artifacts.


In [ ]:
import joblib

models_dir = ROOT / "models"
models_dir.mkdir(parents=True, exist_ok=True)
model_path = models_dir / "baseline_pipeline.joblib"
joblib.dump(pipeline, model_path)
print(f"Saved pipeline to {model_path}")